In [ ]:
import altair as alt
import matplotlib.pyplot as plt
import polars as pl

from traffic_balve.create_df import create_df

In [ ]:
df = (
    create_df()
    .sort("datetime")
    .with_columns(
        delay_due_to_traffic_min=(
            pl.col("duration_in_traffic_s") - pl.col("duration_s")
        )
        / 60,
        delay_due_to_traffic_percent=100
        * (pl.col("duration_in_traffic_s") - pl.col("duration_s"))
        / pl.col("duration_s"),
    )
    .sort("datetime")
    .rolling(
        index_column="datetime",
        period="1m",
        by=["from_to"],
    )
    .agg(
        pl.col(
            "delay_due_to_traffic_percent",
            "delay_due_to_traffic_min",
            "duration_in_traffic_s",
            "duration_s",
        ).mean()
    )
    # .with_columns(
    #     date=pl.col("datetime").dt.date(),
    #     time=pl.col("datetime").dt.time(),
    # )
)
print(df.shape)
df.head(5)

In [ ]:
alt.data_transformers.disable_max_rows()


base = (
    alt.Chart(
        df  # type:ignore
    )
    .mark_line(strokeWidth=3, point=True)
    .encode(
        x=alt.X("hoursminutes(datetime):O").title("Uhrzeit"),
        detail=alt.Detail("monthdate(datetime):O"),
        color=alt.Color("from_to:N")
        .title("Richtung")
        .scale(
            domain=[
                "Höhle -> Krankenhaus",
                "Krankenhaus -> Höhle",
                "Höhle -> Krumpaul",
                "Krumpaul -> Höhle",
                "Krankenhaus -> Krumpaul",
                "Krumpaul -> Krankenhaus",
            ],
            range=[
                "#1F77B4",
                "#AEC7E8",
                "#FF7F0E",
                "#FFBB78",
                "#2CA02C",
                "#98DF8A",
            ],
        ),
    )
    .properties(width=1500, height=250)
)


# alt.layer(
#     base.encode(
#         y=alt.Y("duration_in_traffic_s:Q").title("Reisezeit [Sekunden]"),
#     ),
#     base.mark_line(strokeWidth=2, strokeDash=[2, 2]).encode(
#         y=alt.Y("duration_s:Q").title("Reisezeit [Sekunden]"),
#     ),
# )
alt.layer(
    base.encode(
        y=alt.Y("duration_in_traffic_s:Q").title("Reisezeit [Sekunden]"),
    ),
).facet(row="from_to:N")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)

tmp = (
    create_df()
    .sort("datetime")
    .select(
        "datetime",
        "from_to",
        "duration_in_traffic_s",
    )
    .pivot(index="datetime", columns="from_to", values="duration_in_traffic_s")
)

axs[0].scatter(
    tmp["Höhle -> Krankenhaus"],
    tmp["Krankenhaus -> Höhle"],
    color="blue",
    alpha=0.2,
)
axs[1].scatter(
    tmp["Höhle -> Krumpaul"],
    tmp["Krumpaul -> Höhle"],
    color="orange",
    alpha=0.2,
)
axs[2].scatter(
    tmp["Krankenhaus -> Krumpaul"],
    tmp["Krumpaul -> Krankenhaus"],
    color="green",
    alpha=0.2,
)

for ax in axs:
    ax.grid()
    ax.plot([100, 300], [100, 300], color="gray", zorder=-1)
axs[0].set(
    xlabel="Höhle -> Krankenhaus",
    ylabel="Krankenhaus -> Höhle",
    xlim=(100, 300),
    ylim=(100, 300),
)
axs[1].set(
    xlabel="Höhle -> Krumpaul",
    ylabel="Krumpaul -> Höhle",
    xlim=(100, 300),
    ylim=(100, 300),
)
axs[2].set(
    xlabel="Krankenhaus -> Krumpaul",
    ylabel="Krumpaul -> Krankenhaus",
    xlim=(100, 300),
    ylim=(100, 300),
)

In [ ]:
base = (
    alt.Chart(
        df.sort("datetime")
        .tail(2000)
        .with_columns(
            delta=(
                pl.col("duration_in_traffic_s") - pl.col("duration_in_traffic_s").min()
            ).over("from_to")
        )
    )
    .encode(
        x="hoursminutes(datetime):T",
        y=alt.Y("delta:Q").scale(zero=False),
        color="day(datetime):N",
        tooltip=["datetime:T", "duration_in_traffic_s:Q"],
    )
    .properties(width=900, height=200)
)

(
    alt.layer(base.mark_point(filled=True))
    .facet(row="from_to:N")
    .resolve_scale(y="independent")
)

In [ ]:
base = (
    alt.Chart(
        df.with_columns(
            timestamp=pl.col("datetime").dt.hour().cast(pl.Int64) * 60
            + (pl.col("datetime").dt.minute().cast(pl.Int64) // 15) * 15
        )
        .sort("timestamp")
        .group_by("timestamp", "from_to", maintain_order=True)
        .agg(
            min=pl.col("duration_in_traffic_s").min(),
            mean=pl.col("duration_in_traffic_s").mean(),
            std=pl.col("duration_in_traffic_s").std(),
            median=pl.col("duration_in_traffic_s").median(),
            max=pl.col("duration_in_traffic_s").max(),
        )
        .with_columns(
            meanplusstd=pl.col("mean") + pl.col("std"),
            meanminusstd=pl.col("mean") - pl.col("std"),
            meanplus2std=pl.col("mean") + 2 * pl.col("std"),
            meanminus2std=pl.col("mean") - 2 * pl.col("std"),
        )
        .to_pandas()
    )
    .encode(
        x="timestamp:Q",
        color="from_to:N",
    )
    .properties(width=1200, height=200)
)


alt.layer(
    base.mark_line().encode(y="min:Q", color=alt.value("gray")),
    base.mark_area(opacity=0.2).encode(
        y="meanminus2std:Q",
        y2="meanplus2std:Q",
    ),
    base.mark_area(opacity=0.3).encode(
        y="meanminusstd:Q",
        y2="meanplusstd:Q",
    ),
    base.mark_line().encode(
        y="mean:Q",
    ),
).facet(row="from_to:N")

In [ ]:
df